# LLM reranking example

This notebook continues from the `result` dataset produced by the README's **End-to-end inference** example. It prepares the prompt and candidate-structure PDF required by `PerplexityReranker`, then demonstrates reranking both without few-shot examples and with three curated examples.

## Before you start

Run the notebook from the repository environment after installing NMR-RISE. The end-to-end inference output should be available at `results/my_inference`, unless the variable `result` is already present in this kernel.

Set the API key outside the notebook so that it is not saved in notebook history:

```bash
export PERPLEXITY_API_KEY="your-api-key"
```

> LLM calls consume provider credits. The default example sends one zero-shot request and one few-shot request, with up to three attempts per request if the returned ranking cannot be parsed.

In [ ]:
import copy
import json
import os
import random
from pathlib import Path

from datasets import load_from_disk

from nmr_rise.utils.llm_rerank import PerplexityReranker, llm_rerank_data_gen


def find_repo_root(start=Path.cwd()):
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "src" / "nmr_rise").is_dir():
            return path
    raise RuntimeError("Could not locate the NMR-RISE repository root.")


ROOT = find_repo_root()
ROOT


## 1. Load the end-to-end inference result

If this notebook is run in the same kernel as the README example, the existing `result` variable is reused. Otherwise, the saved Hugging Face dataset is loaded from disk.

In [ ]:
if "result" not in globals():
    result_path = ROOT / "results" / "my_inference"
    if not result_path.exists():
        raise FileNotFoundError(
            f"{result_path} does not exist. Run the README's End-to-end inference "
            "example first, including result.save_to_disk('results/my_inference')."
        )
    result = load_from_disk(str(result_path))

print(result)
print(f"Number of inference entries: {len(result)}")


## 2. Prepare one result for the LLM

`llm_rerank_data_gen` converts an NMR-RISE result entry into a text prompt and a PDF containing its indexed candidate structures. The `smiles` value is used only when evaluating known benchmark targets; it may be empty for a real unknown molecule.

In [ ]:
result_index = 0
rerank_entry = dict(result[result_index])
rerank_entry.setdefault("dataset_idx", 0)
rerank_entry.setdefault("idx", result_index)
rerank_entry.setdefault("smiles", "")

rerank_root = ROOT / "results" / "my_inference" / "llm_rerank"
llm_rerank_data_gen(
    rerank_entry,
    prompt_template_path=str(ROOT / "prompt" / "perplexity_rerank_rmse.txt"),
    save_path=str(rerank_root),
)

case_dir = rerank_root / f"{rerank_entry['dataset_idx']}_{rerank_entry['idx']}"
request = json.loads((case_dir / "prompt.json").read_text(encoding="utf-8"))
request["mol_image_pdf_path"] = str(case_dir / "candidate_structure_images.pdf")

print(f"Prompt: {case_dir / 'prompt.json'}")
print(f"Candidate PDF: {request['mol_image_pdf_path']}")
print(f"Candidates: {len(request['candidates_smiles'])}")


In [ ]:
api_key = os.environ.get("PERPLEXITY_API_KEY")
if not api_key:
    raise RuntimeError(
        "PERPLEXITY_API_KEY is not set. Export it in the shell before starting Jupyter."
    )


## 3. Rerank without few-shot examples

With `use_history_conversation=False`, the model receives only the current NMR case and its candidate PDF. Keep `check_target=False` for unknown structures.

In [ ]:
zero_shot_reranker = PerplexityReranker(
    api_tokens=[api_key],
    max_workers=1,
    timeout=300,
)

zero_shot_result = zero_shot_reranker.llm_rerank(
    entry=copy.deepcopy(request),
    llm_type="sonar-reasoning-pro",
    trials=3,
    check_target=False,
    use_history_conversation=False,
    history_conversation_num=0,
)

print("Reranking succeeded:", zero_shot_result["is_reranked"])
print("Zero-shot ranking:")
for rank, smiles in enumerate(zero_shot_result["reranked_smiles"], start=1):
    print(f"{rank:>2}. {smiles}")


## 4. Load the curated few-shot pool

`data/NMRExp/LLM_Rerank/50` contains **50 manually selected, high-quality reasoning cases** from NMRExp. They were chosen as useful demonstrations of spectrum interpretation, candidate-by-candidate analysis, and evidence-based reranking; they are not generated from the current query. Each case under `perplexity/` provides its prompt, candidate PDF, and curated reranking response.

If the pool is missing, download it from the project repository on Hugging Face:

```bash
hf download Napister/NMR-RISE \
  --repo-type model \
  --include "data/NMRExp/LLM_Rerank/50/**" \
  --local-dir .
```

In [ ]:
few_shot_root = ROOT / "data" / "NMRExp" / "LLM_Rerank" / "50" / "perplexity"
if not few_shot_root.is_dir():
    raise FileNotFoundError(
        f"Few-shot pool not found at {few_shot_root}. Use the download command above."
    )

history_conversation_pool = []
for example_dir in sorted(path for path in few_shot_root.iterdir() if path.is_dir()):
    result_file = example_dir / "reranked_result.json"
    pdf_file = example_dir / "candidate_structure_images.pdf"
    if not result_file.is_file() or not pdf_file.is_file():
        continue

    example = json.loads(result_file.read_text(encoding="utf-8"))
    if not example.get("result_text"):
        continue

    history_conversation_pool.append(
        {
            "idx": example.get("idx", example_dir.name),
            "prompt": example["prompt"],
            "mol_image_pdf_path": str(pdf_file),
            "result_text": example["result_text"],
        }
    )

if len(history_conversation_pool) < 3:
    raise RuntimeError("At least three complete few-shot cases are required.")

print(f"Loaded {len(history_conversation_pool)} curated few-shot cases.")


## 5. Rerank with three few-shot examples

The reranker samples three demonstrations from the curated pool and places them before the current query. Setting the random seed makes this sampling reproducible when using one worker.

In [ ]:
few_shot_reranker = PerplexityReranker(
    api_tokens=[api_key],
    max_workers=1,
    history_conversation_pool=history_conversation_pool,
    timeout=300,
)

random.seed(42)
few_shot_result = few_shot_reranker.llm_rerank(
    entry=copy.deepcopy(request),
    llm_type="sonar-reasoning-pro",
    trials=3,
    check_target=False,
    use_history_conversation=True,
    history_conversation_num=3,
)

print("Reranking succeeded:", few_shot_result["is_reranked"])
print("Few-shot ranking:")
for rank, smiles in enumerate(few_shot_result["reranked_smiles"], start=1):
    print(f"{rank:>2}. {smiles}")


## 6. Compare the rankings

The full chemical reasoning is stored in each result's `result_text`; the cell below compares the candidate order returned by the expert pipeline, zero-shot LLM reranking, and few-shot LLM reranking.

In [ ]:
rankings = {
    "NMR-RISE": request["candidates_smiles"],
    "zero-shot LLM": zero_shot_result["reranked_smiles"],
    "3-shot LLM": few_shot_result["reranked_smiles"],
}

for name, ranking in rankings.items():
    print(f"\n{name}")
    for rank, smiles in enumerate(ranking, start=1):
        print(f"{rank:>2}. {smiles}")

# Inspect the complete reasoning when needed:
# print(few_shot_result["result_text"])


For multiple prepared requests, call `PerplexityReranker.llm_rerank_multiprocess(...)` with the same `use_history_conversation` and `history_conversation_num` settings. LLM reranking remains optional: the NMR-RISE expert-model pipeline already returns a complete ranked candidate list.